# Análisis de ventas con SQL

## Introducción

Este proyecto utiliza SQL para analizar las transacciones de una tienda online británica a partir del conjunto de datos Online Retail.

Los datos contienen líneas de facturas: cada registro representa un producto o concepto incluido en una operación. Una factura puede ocupar varias filas. El conjunto contiene ventas, cancelaciones y otros movimientos.

Construiremos una base de datos SQLite y utilizaremos consultas para explorar las ventas, comparar productos y países, y estudiar la frecuencia de compra de los clientes.

## Preguntas del análisis

- ¿Cómo cambia el importe neto registrado por mes?
- ¿Qué productos aportan más al importe de cada país?
- ¿Qué clientes realizan más compras y cuánto acumulan?
- ¿Qué proporción de clientes compra en más de un mes?

## Herramientas

Utilizaremos SQL para realizar las consultas y Python para cargar el archivo, conectarnos a SQLite y presentar los resultados.

## Fuente de los datos

Chen, D. (2015). Online Retail. UCI Machine Learning Repository.
https://doi.org/10.24432/C5BW33

El archivo `Online Retail.xlsx` debe estar en la misma carpeta que este notebook.

## 1. Carga del archivo y revisión inicial

Cargamos el archivo original y revisamos su tamaño, los tipos de datos y los valores faltantes antes de incorporarlo a SQLite.

Los números de factura y los códigos de producto se leen como texto porque son identificadores y pueden contener letras. El identificador de cliente se representa como un entero que admite valores faltantes.

En esta etapa conservamos todos los registros. Definiremos los criterios del análisis después de revisar su contenido.

In [1]:
import sqlite3
import pandas as pd

datos = pd.read_excel(
    "Online Retail.xlsx",
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "CustomerID": "Int64"
    }
)

print("Número de registros:", len(datos))
print("Número de columnas:", datos.shape[1])

display(datos.head())

resumen = pd.DataFrame({
    "Tipo de dato": datos.dtypes.astype(str),
    "Valores faltantes": datos.isna().sum()
})

display(resumen)

print("Primera fecha:", datos["InvoiceDate"].min())
print("Última fecha:", datos["InvoiceDate"].max())
print("Versión de SQLite:", sqlite3.sqlite_version)

Número de registros: 541909
Número de columnas: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


,Tipo de dato,Valores faltantes
InvoiceNo,string,0
StockCode,string,0
Description,object,1454
Quantity,int64,0
InvoiceDate,datetime64[ns],0
UnitPrice,float64,0
CustomerID,Int64,135080
Country,object,0


Primera fecha: 2010-12-01 08:26:00
Última fecha: 2011-12-09 12:50:00
Versión de SQLite: 3.53.1


## 2. Creación de la base de datos

Guardamos los registros en una base de datos SQLite llamada `online_retail.db`, dentro de la tabla `transacciones`.

Cada fila conserva una línea del archivo original. Todavía no aplicamos filtros ni eliminamos repeticiones.

Los identificadores se almacenan como texto o enteros, según corresponda. Las fechas se guardan como texto en formato año-mes-día y hora, que permite ordenarlas y agruparlas mediante las funciones de SQLite. Los valores faltantes se representan como NULL.

La tabla se reemplaza cada vez que ejecutamos esta celda para evitar acumular los mismos registros en ejecuciones sucesivas.

In [2]:
conexion = sqlite3.connect("online_retail.db")

datos.to_sql(
    "transacciones",
    conexion,
    if_exists="replace",
    index=False,
    dtype={
        "InvoiceNo": "TEXT",
        "StockCode": "TEXT",
        "Description": "TEXT",
        "Quantity": "INTEGER",
        "InvoiceDate": "TEXT",
        "UnitPrice": "REAL",
        "CustomerID": "INTEGER",
        "Country": "TEXT"
    }
)

print("Base de datos creada: online_retail.db")

# Revisamos cómo quedaron definidas las columnas.
estructura = pd.read_sql_query(
    "PRAGMA table_info(transacciones);",
    conexion
)

display(estructura[["name", "type"]])

Base de datos creada: online_retail.db


,name,type
0,InvoiceNo,TEXT
1,StockCode,TEXT
2,Description,TEXT
3,Quantity,INTEGER
4,InvoiceDate,TEXT
5,UnitPrice,REAL
6,CustomerID,INTEGER
7,Country,TEXT


## 3. Primera consulta: comprobación de los registros

Consultamos la tabla para verificar que la carga conserve el número de registros y el período del archivo original.

También contamos los números de factura distintos y los registros sin descripción o identificación de cliente.

COUNT(*) cuenta todas las filas. COUNT(columna) cuenta únicamente los valores que no son NULL. La diferencia entre ambos permite calcular cuántos valores faltan.

El número de facturas distintas incluye todos los tipos de movimientos presentes; todavía no representa un conteo de compras.

In [3]:
consulta = """
SELECT
    COUNT(*) AS Registros,
    COUNT(DISTINCT InvoiceNo) AS Facturas_distintas,
    COUNT(*) - COUNT(Description) AS Sin_descripcion,
    COUNT(*) - COUNT(CustomerID) AS Sin_cliente,
    MIN(InvoiceDate) AS Primera_fecha,
    MAX(InvoiceDate) AS Ultima_fecha
FROM transacciones;
"""

resumen_sql = pd.read_sql_query(consulta, conexion)

display(resumen_sql)

registros_sql = resumen_sql.loc[0, "Registros"]

print("¿Coincide el número de registros?", registros_sql == len(datos))

,Registros,Facturas_distintas,Sin_descripcion,Sin_cliente,Primera_fecha,Ultima_fecha
0,541909,25900,1454,135080,2010-12-01 08:26:00,2011-12-09 12:50:00


¿Coincide el número de registros? True


## 4. Selección de los movimientos de productos

Creamos una vista llamada `movimientos_productos` con los registros incluidos en el análisis. Una vista guarda una consulta y permite utilizar su resultado como una tabla, sin duplicar los datos.

Aplicamos estos criterios:

- Conservar precios positivos y cantidades distintas de cero.
- Excluir códigos identificados como envíos, comisiones, descuentos, ajustes contables, operaciones manuales, muestras y vales de regalo.
- Mantener las cantidades negativas para calcular el importe neto registrado.
- Conservar los registros sin cliente identificado para los análisis generales.
- Mantener las filas repetidas, porque no contamos con un identificador de línea que permita confirmar que sean errores.

Normalizamos los códigos de factura y producto eliminando espacios externos y convirtiendo las letras a mayúsculas.

El importe de cada movimiento se calcula multiplicando cantidad por precio unitario. Su suma representa el importe neto registrado de los productos seleccionados; no corresponde a ganancias ni incluye todos los conceptos de las facturas.

In [4]:
consulta = """
CREATE VIEW IF NOT EXISTS movimientos_productos AS
SELECT
    UPPER(TRIM(InvoiceNo)) AS Factura,
    UPPER(TRIM(StockCode)) AS Producto,
    Description AS Descripcion,
    Quantity AS Cantidad,
    InvoiceDate AS Fecha,
    UnitPrice AS Precio_unitario,
    CustomerID AS Cliente,
    Country AS Pais,
    Quantity * UnitPrice AS Importe,
    CASE
        WHEN UPPER(TRIM(InvoiceNo)) LIKE 'C%' THEN 'Cancelación'
        WHEN Quantity < 0 THEN 'Otro movimiento negativo'
        ELSE 'Venta positiva'
    END AS Tipo_movimiento
FROM transacciones
WHERE UnitPrice > 0
    AND Quantity <> 0
    AND UPPER(TRIM(StockCode)) NOT IN (
        'AMAZONFEE', 'B', 'BANK CHARGES', 'C2', 'CRUK',
        'D', 'DOT', 'M', 'POST', 'S'
    )
    AND SUBSTR(UPPER(TRIM(StockCode)), 1, 5) <> 'GIFT_';
"""

# Recreamos la vista para aplicar cualquier cambio en sus criterios.
conexion.execute("DROP VIEW IF EXISTS movimientos_productos;")
conexion.execute(consulta)
conexion.commit()

print("Vista creada: movimientos_productos")

Vista creada: movimientos_productos


## 5. Comprobación de la selección

Contamos los registros incluidos y excluidos, y revisamos los importes por tipo de movimiento.

La clasificación utiliza primero el código de factura para identificar cancelaciones. Las cantidades negativas sin ese código se muestran por separado, sin asumir que todas correspondan a devoluciones.

Las ventas positivas representan importes anteriores a posibles cancelaciones posteriores. El importe neto suma todos los movimientos seleccionados con su signo.

In [5]:
consulta = """
SELECT
    (SELECT COUNT(*) FROM transacciones) AS Registros_originales,
    COUNT(*) AS Movimientos_seleccionados,
    (SELECT COUNT(*) FROM transacciones) - COUNT(*) AS Registros_excluidos,
    COUNT(*) - COUNT(Cliente) AS Registros_sin_cliente,
    ROUND(SUM(Importe), 2) AS Importe_neto_GBP
FROM movimientos_productos;
"""

display(pd.read_sql_query(consulta, conexion))

consulta = """
SELECT
    Tipo_movimiento,
    COUNT(*) AS Registros,
    SUM(Cantidad) AS Unidades,
    ROUND(SUM(Importe), 2) AS Importe_GBP
FROM movimientos_productos
GROUP BY Tipo_movimiento
ORDER BY Importe_GBP DESC;
"""

display(pd.read_sql_query(consulta, conexion))

,Registros_originales,Movimientos_seleccionados,Registros_excluidos,Registros_sin_cliente,Importe_neto_GBP
0,541909,536465,5444,131586,9792708.88


,Tipo_movimiento,Registros,Unidades,Importe_GBP
0,Venta positiva,527761,5577103,10271433.06
1,Cancelación,8704,-272030,-478724.18


### Resultado de la selección

Conservamos 536.465 movimientos y excluimos 5.444 registros según los criterios establecidos.

Las ventas positivas suman £10.271.433,06 y las cancelaciones suman −£478.724,18. El importe neto registrado es de £9.792.708,88.

Los 131.586 registros sin cliente identificado permanecen en el análisis general. No hay movimientos con cantidades negativas sin código de cancelación dentro de la selección.

Las cancelaciones se contabilizan en la fecha en que fueron registradas. No se reasignan al mes de la venta original.

## 6. Evolución mensual del importe

Agrupamos los movimientos por año y mes para calcular las ventas positivas, las cancelaciones y el importe neto registrado.

También contamos las facturas con al menos una línea de venta positiva seleccionada. Las facturas de cancelación no forman parte de ese conteo.

Utilizamos una CTE para organizar el resumen mensual y la función LAG para obtener el importe neto del mes anterior. Con ambos valores calculamos la variación porcentual.

Diciembre de 2011 contiene datos únicamente hasta el día 9. Mostramos sus importes, pero no calculamos su variación frente a noviembre porque los períodos tienen distinta cobertura. El primer mes tampoco tiene una variación disponible.

In [6]:
consulta = """
WITH resumen_mensual AS (
    SELECT
        STRFTIME('%Y-%m', Fecha) AS Mes,
        COUNT(DISTINCT CASE
            WHEN Tipo_movimiento = 'Venta positiva' THEN Factura
        END) AS Facturas_venta,
        SUM(CASE
            WHEN Tipo_movimiento = 'Venta positiva' THEN Importe
            ELSE 0
        END) AS Ventas_positivas,
        SUM(CASE
            WHEN Tipo_movimiento = 'Cancelación' THEN Importe
            ELSE 0
        END) AS Cancelaciones,
        SUM(Importe) AS Importe_neto
    FROM movimientos_productos
    GROUP BY STRFTIME('%Y-%m', Fecha)
),
comparacion_mensual AS (
    SELECT
        *,
        LAG(Importe_neto) OVER (ORDER BY Mes) AS Neto_mes_anterior
    FROM resumen_mensual
)
SELECT
    Mes,
    Facturas_venta,
    ROUND(Ventas_positivas, 2) AS Ventas_positivas_GBP,
    ROUND(Cancelaciones, 2) AS Cancelaciones_GBP,
    ROUND(Importe_neto, 2) AS Importe_neto_GBP,
    CASE
        WHEN Mes = '2011-12' THEN NULL
        ELSE ROUND(
            100.0 * (Importe_neto - Neto_mes_anterior)
            / NULLIF(Neto_mes_anterior, 0),
            2
        )
    END AS Variacion_neta_pct,
    CASE
        WHEN Mes = '2011-12' THEN 'Parcial: hasta el día 9'
        ELSE 'Mes completo dentro del período'
    END AS Cobertura
FROM comparacion_mensual
ORDER BY Mes;
"""

resumen_mensual = pd.read_sql_query(consulta, conexion)

display(resumen_mensual)

,Mes,Facturas_venta,Ventas_positivas_GBP,Cancelaciones_GBP,Importe_neto_GBP,Variacion_neta_pct,Cobertura
0,2010-12,1550,777931.77,-17607.86,760323.91,NaN,Mes completo dentro del período
1,2011-01,1081,671933.19,-91526.30,580406.89,-23.66,Mes completo dentro del período
2,2011-02,1093,508886.20,-8375.87,500510.33,-13.77,Mes completo dentro del período
3,2011-03,1440,691257.61,-10675.30,680582.31,35.98,Mes completo dentro del período
4,2011-04,1235,516218.12,-33316.91,482901.21,-29.05,Mes completo dentro del período
5,2011-05,1668,741192.41,-8948.23,732244.18,51.63,Mes completo dentro del período
6,2011-06,1525,738751.87,-13760.84,724991.03,-0.99,Mes completo dentro del período
7,2011-07,1452,689364.41,-11409.21,677955.20,-6.49,Mes completo dentro del período
8,2011-08,1339,725513.50,-22929.52,702583.98,3.63,Mes completo dentro del período
9,2011-09,1818,1030475.36,-17044.68,1013430.68,44.24,Mes completo dentro del período


### Interpretación de la evolución mensual

Noviembre de 2011 presenta el mayor importe neto entre los meses completos, con £1.432.734,99 y 2.751 facturas de venta.

El mayor aumento porcentual mensual ocurre en mayo, con un 51,63 % respecto de abril. Este crecimiento sucede después de una caída del 29,05 %, por lo que debe interpretarse considerando la base de comparación.

Entre agosto y noviembre, el importe neto aumenta cada mes. Sin embargo, el período disponible no permite confirmar un patrón estacional que se repita entre años.

Diciembre de 2011 contiene registros únicamente hasta el día 9. Su importe no debe compararse directamente con el total de noviembre para concluir que las ventas disminuyeron.

Las cancelaciones reducen el importe del mes en que fueron registradas, aunque la venta original podría pertenecer a otro mes.

## 7. Productos con mayor importe neto por país

Calculamos el importe neto de cada producto dentro de cada país y seleccionamos los tres primeros puestos.

Utilizamos DENSE_RANK para ordenar los productos por su importe neto dentro de cada país. Si existen empates, los productos comparten posición, por lo que un país podría mostrar más de tres productos.

Mediante JOIN incorporamos el importe neto total del país y calculamos la participación de cada producto sobre ese total.

Agrupamos por código de producto, porque las descripciones pueden variar. El ranking incluye únicamente productos con importe neto positivo y países cuyo importe neto total también es positivo.

La participación utiliza como denominador todos los movimientos seleccionados del país, incluidos los negativos. Describe la contribución al importe neto, no la rentabilidad.

Los puestos se calculan con importes redondeados a dos decimales para conservar los empates monetarios.

In [7]:
consulta = """
WITH productos_por_pais AS (
    SELECT
        Pais,
        Producto,
        SUM(Cantidad) AS Unidades_netas,
        SUM(Importe) AS Importe_neto_producto
    FROM movimientos_productos
    GROUP BY Pais, Producto
),
totales_por_pais AS (
    SELECT
        Pais,
        SUM(Importe) AS Importe_neto_pais
    FROM movimientos_productos
    GROUP BY Pais
),
ranking AS (
    SELECT
        Pais,
        Producto,
        Unidades_netas,
        Importe_neto_producto,
        DENSE_RANK() OVER (
            PARTITION BY Pais
            ORDER BY ROUND(Importe_neto_producto, 2) DESC
        ) AS Posicion
    FROM productos_por_pais
    WHERE Importe_neto_producto > 0
)
SELECT
    r.Pais,
    r.Posicion,
    r.Producto,
    r.Unidades_netas,
    ROUND(r.Importe_neto_producto, 2) AS Importe_neto_GBP,
    ROUND(
        100.0 * r.Importe_neto_producto / t.Importe_neto_pais,
        2
    ) AS Participacion_pais_pct
FROM ranking AS r
JOIN totales_por_pais AS t
    ON r.Pais = t.Pais
WHERE r.Posicion <= 3
    AND t.Importe_neto_pais > 0
ORDER BY r.Pais, r.Posicion, r.Producto;
"""

ranking_productos = pd.read_sql_query(consulta, conexion)

display(ranking_productos.head(15))

print("Países incluidos:", ranking_productos["Pais"].nunique())
print("Filas del ranking:", len(ranking_productos))

,Pais,Posicion,Producto,Unidades_netas,Importe_neto_GBP,Participacion_pais_pct
0,Australia,1,23084,1884,3375.84,2.46
1,Australia,2,22722,600,2082.00,1.52
2,Australia,3,21731,1344,1987.20,1.45
3,Austria,1,22582,144,302.40,3.48
4,Austria,1,22584,144,302.40,3.48
5,Austria,2,15056BL,36,214.20,2.46
6,Austria,2,15056N,36,214.20,2.46
7,Austria,2,20679,36,214.20,2.46
8,Austria,3,22139,45,205.95,2.37
9,Bahrain,1,23076,96,120.00,21.88


Países incluidos: 38
Filas del ranking: 134


### Interpretación del ranking de productos

El ranking incluye 38 países y 134 filas. Al conservar los empates, algunos países presentan más de tres productos entre las primeras tres posiciones.

En Australia, el producto 23084 ocupa el primer puesto con £3.375,84 y representa el 2,46 % del importe neto del país.

En Austria, los productos 22582 y 22584 comparten el primer puesto, ambos con £302,40.

En Bahrain, el producto 23076 representa el 21,88 % del importe neto del país, aunque su importe es de solo £120. Esto muestra que una participación elevada dentro de un mercado no implica un importe elevado en términos absolutos.

El ranking describe la contribución monetaria de los productos dentro de cada país. No permite comparar su rentabilidad, porque no contamos con información de costos.

## 8. Clientes con mayor número de facturas de venta

Agrupamos los movimientos por cliente y contamos las facturas distintas que contienen al menos una línea de venta positiva seleccionada.

Calculamos también las ventas positivas, las cancelaciones y el importe neto registrado. Excluimos de esta consulta los movimientos sin identificación de cliente.

Una factura con varios productos cuenta como una sola compra. Las facturas de cancelación no aumentan el conteo de compras.

Mostramos los diez clientes con más facturas de venta. En caso de empate, ordenamos por importe neto y luego por identificador. La frecuencia y el importe acumulado describen aspectos diferentes del comportamiento de compra.

In [8]:
consulta = """
SELECT
    Cliente,
    COUNT(DISTINCT CASE
        WHEN Tipo_movimiento = 'Venta positiva' THEN Factura
    END) AS Facturas_venta,
    COUNT(DISTINCT CASE
        WHEN Tipo_movimiento = 'Venta positiva'
        THEN STRFTIME('%Y-%m', Fecha)
    END) AS Meses_con_compras,
    ROUND(SUM(CASE
        WHEN Tipo_movimiento = 'Venta positiva' THEN Importe
        ELSE 0
    END), 2) AS Ventas_positivas_GBP,
    ROUND(SUM(CASE
        WHEN Tipo_movimiento = 'Cancelación' THEN Importe
        ELSE 0
    END), 2) AS Cancelaciones_GBP,
    ROUND(SUM(Importe), 2) AS Importe_neto_GBP
FROM movimientos_productos
WHERE Cliente IS NOT NULL
GROUP BY Cliente
HAVING COUNT(DISTINCT CASE
    WHEN Tipo_movimiento = 'Venta positiva' THEN Factura
END) > 0
ORDER BY Facturas_venta DESC, Importe_neto_GBP DESC, Cliente
LIMIT 10;
"""

clientes_frecuentes = pd.read_sql_query(consulta, conexion)

display(clientes_frecuentes)

,Cliente,Facturas_venta,Meses_con_compras,Ventas_positivas_GBP,Cancelaciones_GBP,Importe_neto_GBP
0,12748,206,13,32317.32,-1830.79,30486.53
1,14911,198,13,136275.72,-7393.59,128882.13
2,17841,124,13,40967.72,-634.50,40333.22
3,13089,97,13,58825.83,-1439.95,57385.88
4,15311,91,13,60767.90,-1348.56,59419.34
5,14606,90,13,12006.65,-439.55,11567.10
6,12971,86,13,11189.91,-256.15,10933.76
7,14646,72,13,279138.02,-360.00,278778.02
8,16029,62,12,72882.09,-14688.24,58193.85
9,13408,62,13,28117.04,-228.66,27888.38


### Interpretación de la frecuencia de compra

El cliente 12748 presenta el mayor número de facturas de venta, con 206 operaciones y un importe neto de £30.486,53.

Dentro de los diez clientes más frecuentes, el cliente 14646 tiene el mayor importe neto, con £278.778,02, aunque registra 72 facturas. La frecuencia de compra y el importe acumulado describen aspectos diferentes del comportamiento del cliente.

Nueve de los diez clientes del ranking realizaron compras en los 13 meses calendario presentes en los datos. Este conteo incluye diciembre de 2011, que contiene registros únicamente hasta el día 9.

Los resultados corresponden a clientes identificados y a movimientos registrados dentro del período. No permiten conocer su historial anterior ni asegurar que continuarán comprando.

## 9. Clientes con compras en distintos meses

Contamos los meses calendario distintos en que cada cliente identificado registra al menos una factura de venta positiva.

Separamos a los clientes en dos grupos: quienes compraron en un solo mes y quienes compraron en dos o más meses. Calculamos la participación de cada grupo sobre el total de clientes con compras seleccionadas.

Comprar en un solo mes no significa necesariamente realizar una sola compra: un cliente puede tener varias facturas dentro del mismo mes.

Este indicador describe la recurrencia observada entre meses. No es una tasa de retención: los clientes tienen distintos tiempos de seguimiento y quienes aparecen cerca del final del período tienen menos oportunidades de registrar compras en meses posteriores.

In [9]:
consulta = """
WITH actividad_clientes AS (
    SELECT
        Cliente,
        COUNT(DISTINCT STRFTIME('%Y-%m', Fecha)) AS Meses_con_compras
    FROM movimientos_productos
    WHERE Cliente IS NOT NULL
        AND Tipo_movimiento = 'Venta positiva'
    GROUP BY Cliente
),
grupos AS (
    SELECT
        CASE
            WHEN Meses_con_compras = 1 THEN 'Un solo mes'
            ELSE 'Dos o más meses'
        END AS Frecuencia,
        COUNT(*) AS Clientes
    FROM actividad_clientes
    GROUP BY CASE
        WHEN Meses_con_compras = 1 THEN 'Un solo mes'
        ELSE 'Dos o más meses'
    END
)
SELECT
    Frecuencia,
    Clientes,
    ROUND(
        100.0 * Clientes / SUM(Clientes) OVER (),
        2
    ) AS Participacion_pct
FROM grupos
ORDER BY Clientes DESC;
"""

recurrencia_clientes = pd.read_sql_query(consulta, conexion)

display(recurrencia_clientes)

print("Total de clientes analizados:", recurrencia_clientes["Clientes"].sum())

,Frecuencia,Clientes,Participacion_pct
0,Dos o más meses,2695,62.18
1,Un solo mes,1639,37.82


Total de clientes analizados: 4334


### Interpretación de la recurrencia entre meses

De los 4.334 clientes identificados con ventas positivas seleccionadas, 2.695 realizaron compras en dos o más meses, equivalentes al 62,18 %. Los otros 1.639 clientes, un 37,82 %, concentraron sus compras en un solo mes.

La mayoría de los clientes analizados presenta compras en distintos meses del período observado. Sin embargo, estos meses no necesariamente son consecutivos.

Este resultado no representa una tasa de retención. No todos los clientes disponen del mismo tiempo de seguimiento y diciembre de 2011 está incompleto. Además, comprar en un único mes no implica haber realizado una sola compra.

## 10. Conclusiones y limitaciones

### Conclusiones

El análisis utiliza SQL para seleccionar movimientos, resumir importes, construir rankings y estudiar la frecuencia de compra.

- Los 536.465 movimientos seleccionados suman un importe neto registrado de £9.792.708,88.
- Noviembre de 2011 presenta el mayor importe neto entre los meses completos, con £1.432.734,99.
- El ranking de productos incluye 38 países y conserva los empates dentro de las primeras tres posiciones.
- La frecuencia de compra y el importe acumulado no son equivalentes: el cliente con más facturas no es quien acumula el mayor importe dentro del ranking analizado.
- El 62,18 % de los clientes identificados con ventas positivas realizó compras en dos o más meses.

### Limitaciones

El importe neto corresponde a los productos incluidos según los criterios de selección. Excluye conceptos como envíos y comisiones, y no representa ganancias.

Las cancelaciones se contabilizan en su fecha de registro, sin vincularlas a la venta original. Por tanto, pueden reducir el importe de un mes distinto al de la compra.

Se conservan las filas exactamente repetidas porque no contamos con un identificador de línea que permita confirmar errores.

Los análisis de clientes excluyen los registros sin identificación. Sus resultados no describen necesariamente todas las compras de la tienda.

Diciembre de 2011 contiene datos hasta el día 9. El período disponible y las diferencias en el tiempo de seguimiento impiden interpretar la recurrencia entre meses como una tasa de retención.

## 11. Exportación de resultados

Exportamos las tablas del análisis como archivos CSV para facilitar su revisión y uso en otras herramientas.

La base de datos `online_retail.db` contiene la tabla original y la vista de movimientos seleccionados. Puede consultarse nuevamente sin repetir la importación del Excel.

Los CSV contienen los resultados de las consultas, con los importes y porcentajes redondeados para su presentación.

In [10]:
from pathlib import Path

carpeta = Path("resultados")
carpeta.mkdir(exist_ok=True)

resumen_mensual.to_csv(carpeta / "resumen_mensual.csv", index=False)
ranking_productos.to_csv(carpeta / "ranking_productos_por_pais.csv", index=False)
clientes_frecuentes.to_csv(carpeta / "clientes_frecuentes.csv", index=False)
recurrencia_clientes.to_csv(carpeta / "recurrencia_clientes.csv", index=False)

print("Resultados guardados en la carpeta resultados.")
print("Base de datos disponible: online_retail.db")

Resultados guardados en la carpeta resultados.
Base de datos disponible: online_retail.db
